# Topic 1 · Regression and uncertainty

**How do you turn an amplitude into a magnitude — and how well do you know the answer?**

The two papers we follow:

- **Richter (1935)**, *BSSA* 25(1), `10.1785/bssa0250010001` — magnitude is invented. The scale is
  defined by a table of distance corrections, so the first magnitude was a fitted curve.
- **Hutton & Boore (1987)**, *BSSA* 77(6), `10.1785/bssa0770062074` — the distance correction
  southern California still runs on, and the coefficients this session measures itself against.

The session is in three parts.

**Part A — the method.** Built on data we generate, from coefficients we choose. Every estimate can
then be checked against a truth that is known, which is the only way to ask whether a stated
interval is honest. Nothing in Part A is a measurement of the Earth.

**Part B — magnitude calibration.** The same design matrix, with real Wood–Anderson amplitudes and
no known truth.

**Part C — the Gutenberg–Richter *b*-value.** The same recipe once more, on data whose noise is not
Gaussian — where least squares stops being the right estimator and says so quietly.

## Part A · Regression and uncertainty, where the answer is known

Everything in this part is generated. We fix a set of coefficients, draw noisy observations from
them, and then estimate the coefficients back — so at every step the estimate can be compared with
the number that produced the data, and every claimed uncertainty can be tested against how often it
is actually right.

The generator is not arbitrary. It is the functional form of a local-magnitude attenuation
relation, with Hutton & Boore's published coefficients used as its truth, so that Part B is the
same fit on real amplitudes rather than a new model to learn.

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(207)

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.axisbelow": True})
print("seed 207 — every number below is reproducible")

### A.1 · A model is a design matrix

A seismic station records a peak amplitude $A$ from an earthquake of magnitude $M$ at hypocentral
distance $R$. The relation seismologists use is

$$\log_{10} A \;=\; c_0 \;+\; c_1 M \;+\; c_2 \log_{10} R \;+\; c_3 R \;+\; \varepsilon .$$

The two distance terms carry different physics: $c_2\log_{10}R$ is geometrical spreading, which
falls off as a power of distance, and $c_3 R$ is anelastic attenuation, which is exponential in
distance and so linear in the log. Both are *non-linear functions of $R$* — and neither of them
makes the model non-linear, because what matters is that the relation is **linear in the
parameters** $c_0 \ldots c_3$. Stack one row per reading:

$$y = X\beta + \varepsilon, \qquad
X = \begin{bmatrix} 1 & M_1 & \log_{10}R_1 & R_1 \\ \vdots & \vdots & \vdots & \vdots \end{bmatrix}$$

That matrix $X$ is the **design matrix**, and the columns are whatever functions of the measured
variables the physics asks for. Recognising that a scientific relation is linear in its parameters
is the transferable move: once it is written this way, every linear model in this course is the same
solve, and the choice of columns is where the seismology lives.

In [ ]:
# The truth. These are Hutton & Boore's (1987) coefficients, rewritten from their
# centred form -log10 A0 = 1.110 log10(R/100) + 0.00189 (R - 100) + 3.0 into the
# uncentred columns above. In Part A they are a definition; in Part B they are the
# published value a real fit is compared against.
BETA_TRUE = np.array([-0.591, 1.000, -1.110, -0.00189])   # c0, c1, c2, c3
SIGMA_TRUE = 0.30                                          # log10 units of scatter per reading
NAMES = ["c0 (const)", "c1 (M)", "c2 (log10 R)", "c3 (R)"]

for nm, b in zip(NAMES, BETA_TRUE):
    print(f"{nm:>14s} = {b:9.5f}")
print(f"{'sigma':>14s} = {SIGMA_TRUE:9.5f}")

In [ ]:
# One synthetic network: 2000 readings, magnitudes 1.0-4.5, distances 5-300 km
# drawn log-uniformly so the near field is not swamped by the far field.
n = 2000
M = rng.uniform(1.0, 4.5, n)
R = 10 ** rng.uniform(np.log10(5), np.log10(300), n)

X = np.column_stack([np.ones(n), M, np.log10(R), R])
y = X @ BETA_TRUE + rng.normal(0, SIGMA_TRUE, n)

print("X shape", X.shape, " -> one row per reading, one column per parameter")
print("first three rows of X:")
print(np.round(X[:3], 4))

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0))
s = ax.scatter(R, y, c=M, s=6, cmap="viridis", alpha=0.6, linewidths=0)
rr = np.logspace(np.log10(5), np.log10(300), 200)
for m in (1.5, 2.5, 3.5, 4.5):
    xx = np.column_stack([np.ones_like(rr), np.full_like(rr, m), np.log10(rr), rr])
    ax.plot(rr, xx @ BETA_TRUE, "k-", lw=1.0)
    ax.text(6, xx[0] @ BETA_TRUE + 0.06, f"M {m}", fontsize=7)
ax.set_xscale("log")
ax.set_xlabel("hypocentral distance R (km)")
ax.set_ylabel(r"$\log_{10} A$")
ax.set_title(f"{n} synthetic readings; black lines are the generator, not a fit")
fig.colorbar(s, ax=ax, label="magnitude")
fig.tight_layout()
plt.show()

### A.2 · Least squares and the normal equations

Choose $\hat\beta$ to minimise the sum of squared residuals,
$S(\beta) = \lVert y - X\beta \rVert^2$.

1. Expand: $S(\beta) = y^\top y - 2\beta^\top X^\top y + \beta^\top X^\top X \beta$.
2. Differentiate with respect to $\beta$:
   $\partial S/\partial\beta = -2X^\top y + 2X^\top X\beta$.
3. Set it to zero. This gives the **normal equations**, $X^\top X\,\hat\beta = X^\top y$, and where
   $X^\top X$ is invertible, $\hat\beta = (X^\top X)^{-1}X^\top y$.
4. The second derivative is $2X^\top X$, which is positive semi-definite for any $X$, so the
   stationary point is a minimum and not a maximum.

Step 3 has a geometric reading worth carrying. Rearranged, it says
$X^\top(y - X\hat\beta) = 0$: the residual vector is **orthogonal to every column of $X$**. Least
squares is the projection of $y$ onto the column space of $X$, and what is left over is, by
construction, the part of the data the model cannot represent. This is why adding a column can only
reduce the residual — and why a reduced residual is not by itself evidence of anything.

In [ ]:
# Two routes to the same estimate. The second is the one to use: lstsq goes through
# a QR/SVD factorisation and does not form X'X, which squares the condition number.
XtX, Xty = X.T @ X, X.T @ y
beta_normal = np.linalg.solve(XtX, Xty)
beta_hat, *_ = np.linalg.lstsq(X, y, rcond=None)

print(f"{'':>14s} {'normal eqns':>12s} {'lstsq':>12s} {'truth':>12s}")
for nm, a, b, t in zip(NAMES, beta_normal, beta_hat, BETA_TRUE):
    print(f"{nm:>14s} {a:12.5f} {b:12.5f} {t:12.5f}")
print(f"\nlargest disagreement between the two routes: {np.abs(beta_normal - beta_hat).max():.2e}")

In [ ]:
# The orthogonality of step 3, checked rather than asserted.
resid = y - X @ beta_hat
print("X' r, one entry per column of X:", np.array2string(X.T @ resid, precision=9))
print(f"largest |X' r| = {np.abs(X.T @ resid).max():.2e}  (zero, to machine precision)")

rss = resid @ resid
p = X.shape[1]
sigma_hat = np.sqrt(rss / (n - p))
print(f"\nsigma_hat = sqrt(RSS/(n-p)) = {sigma_hat:.4f}   against a true {SIGMA_TRUE}")

### A.3 · Least squares is maximum likelihood under Gaussian noise

Nothing so far said anything about probability: we minimised a sum of squares because it was a
convenient thing to minimise. Now assume the errors are independent and Gaussian,
$\varepsilon_i \sim N(0,\sigma^2)$, and the choice stops being a convention.

1. The density of one observation is
   $p(y_i \mid \beta,\sigma) = (2\pi\sigma^2)^{-1/2}\exp\!\big[-(y_i - x_i^\top\beta)^2/2\sigma^2\big]$.
2. Independence makes the likelihood the product,
   $L(\beta,\sigma) = \prod_i p(y_i\mid\beta,\sigma)$.
3. Take the log, which is monotone and so does not move the maximum:
   $\ell(\beta,\sigma) = -\tfrac{n}{2}\log(2\pi\sigma^2) - \dfrac{1}{2\sigma^2}\lVert y - X\beta\rVert^2$.
4. Only the last term contains $\beta$, and it is $-S(\beta)/2\sigma^2$ — a negative constant times
   the sum of squares. **Maximising $\ell$ over $\beta$ is therefore exactly minimising
   $S(\beta)$**, whatever $\sigma$ happens to be.

So least squares is not a definition; it is the maximum-likelihood estimator *for one particular
noise model*. That distinction is the whole of Part C.

It also gives the recipe this session uses three times:

> **(i)** write down the likelihood of the data under the model;
> **(ii)** maximise it to get the estimate;
> **(iii)** take the curvature of $\ell$ at the maximum to get the uncertainty.

Steps (i) and (iii) never mention Gaussians. Only step (ii) collapses into the normal equations, and
only when the noise is Gaussian.

In [ ]:
def loglik(b, sig):
    r = y - X @ b
    return -0.5 * n * np.log(2 * np.pi * sig**2) - (r @ r) / (2 * sig**2)

# Slice the log-likelihood along c1 with the other three held at their fitted values,
# and overlay the sum of squares rescaled by -1/(2 sigma^2). Step 4 says they coincide.
c1_grid = np.linspace(beta_hat[1] - 0.05, beta_hat[1] + 0.05, 201)
ll = np.array([loglik(np.r_[beta_hat[0], c, beta_hat[2:]], sigma_hat) for c in c1_grid])
sse = np.array([((y - X @ np.r_[beta_hat[0], c, beta_hat[2:]])**2).sum() for c in c1_grid])

print(f"c1 maximising the log-likelihood: {c1_grid[ll.argmax()]:.5f}")
print(f"c1 minimising the sum of squares: {c1_grid[sse.argmin()]:.5f}")
print(f"c1 from lstsq:                    {beta_hat[1]:.5f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.4))
a1.plot(c1_grid, ll, lw=1.6, label="log-likelihood")
a1.plot(c1_grid, -sse / (2 * sigma_hat**2) + (ll.max() + sse.min() / (2 * sigma_hat**2)),
        "--", lw=1.6, label=r"$-S(\beta)/2\sigma^2$ + const")
a1.axvline(beta_hat[1], color="k", lw=0.8)
a1.set_xlabel(r"$c_1$"); a1.set_ylabel("value"); a1.legend(fontsize=7)
a1.set_title("same maximiser, two derivations")

g2, g3 = np.meshgrid(np.linspace(-1.35, -0.90, 90), np.linspace(-0.0035, -0.0005, 90))
Z = np.array([loglik(np.r_[beta_hat[:2], u, v], sigma_hat)
              for u, v in zip(g2.ravel(), g3.ravel())]).reshape(g2.shape)
a2.contour(g2, g3, Z, levels=np.percentile(Z, [88, 94, 97, 99, 99.7]), linewidths=0.8)
a2.plot(*beta_hat[2:], "r+", ms=10)
a2.set_xlabel(r"$c_2$"); a2.set_ylabel(r"$c_3$")
a2.set_title("log-likelihood in the two distance terms")
fig.tight_layout(); plt.show()

### A.4 · Where the error bars come from

$\hat\beta$ is a function of noisy data, so it is itself a random variable. Substitute
$y = X\beta + \varepsilon$ into the estimator:

$$\hat\beta = (X^\top X)^{-1}X^\top(X\beta + \varepsilon) = \beta + (X^\top X)^{-1}X^\top\varepsilon .$$

1. The second term has mean zero, so $\hat\beta$ is **unbiased**: on average it is the truth.
2. Its covariance is
   $\operatorname{Cov}(\hat\beta) = (X^\top X)^{-1}X^\top \operatorname{Cov}(\varepsilon) X (X^\top X)^{-1}$,
   and with $\operatorname{Cov}(\varepsilon) = \sigma^2 I$ the middle collapses and this is
   $\boxed{\sigma^2 (X^\top X)^{-1}}$.
3. $\sigma$ is unknown, so use $\hat\sigma^2 = \mathrm{RSS}/(n-p)$; the divisor is $n-p$ rather than
   $n$ because $p$ directions of the residual were spent fitting.

Now connect it to step (iii) of the recipe. Differentiate $\ell$ from A.3 twice:
$-\partial^2\ell/\partial\beta\partial\beta^\top = X^\top X/\sigma^2$. That is the **Fisher
information** — how sharply peaked the likelihood is — and $\operatorname{Cov}(\hat\beta)$ is its
inverse. A sharp peak is a well-determined parameter.

This is the sentence that survives into Part C: *the uncertainty is the inverse curvature of the
log-likelihood at its maximum.* The Gaussian case is the one where that curvature happens to have
the closed form $\sigma^2(X^\top X)^{-1}$.

In [ ]:
Cov = sigma_hat**2 * np.linalg.inv(XtX)
se = np.sqrt(np.diag(Cov))

print(f"{'':>14s} {'estimate':>11s} {'std error':>11s} {'truth':>11s} {'(est-truth)/se':>15s}")
for nm, b, s_, t in zip(NAMES, beta_hat, se, BETA_TRUE):
    print(f"{nm:>14s} {b:11.5f} {s_:11.5f} {t:11.5f} {(b - t) / s_:15.2f}")

fisher = XtX / sigma_hat**2
print(f"\nlargest |Cov - inverse Fisher information| = "
      f"{np.abs(Cov - np.linalg.inv(fisher)).max():.2e}")

**Exercise 1.** A standard error is a claim about repetition: over many repeats of the
experiment, the interval $\hat\beta_j \pm 1.96\,\mathrm{se}_j$ should contain the true $c_j$ about
95 % of the time. That claim is testable here and nowhere in Parts B or C, because only here is the
truth known.

Regenerate `y` from `BETA_TRUE` 1000 times using the same design matrix `X`, refit each time, and
report — for each coefficient — the fraction of refits whose interval covered the truth.

In [ ]:
# your code here


In [ ]:
# ── Checkpoint 1 ── run this if you are behind or something broke ──
# State Part A needs from here on: the design, the fit, and its covariance.
n, p = 2000, 4
BETA_TRUE = np.array([-0.591, 1.000, -1.110, -0.00189]); SIGMA_TRUE = 0.30
_r = np.random.default_rng(207)
M = _r.uniform(1.0, 4.5, n); R = 10 ** _r.uniform(np.log10(5), np.log10(300), n)
X = np.column_stack([np.ones(n), M, np.log10(R), R])
y = X @ BETA_TRUE + _r.normal(0, SIGMA_TRUE, n)
XtX = X.T @ X
beta_hat, *_ = np.linalg.lstsq(X, y, rcond=None)
resid = y - X @ beta_hat
sigma_hat = np.sqrt(resid @ resid / (n - p))
Cov = sigma_hat**2 * np.linalg.inv(XtX)
se = np.sqrt(np.diag(Cov))
print("recovered:", np.round(beta_hat, 4))

### A.5 · Confidence interval versus prediction interval

Two questions get asked of a fitted curve, and they are not the same question.

**Where does the curve go?** The fitted value at a new row $x_0$ is $x_0^\top\hat\beta$, a linear
function of $\hat\beta$, so from A.4

$$\operatorname{Var}(x_0^\top\hat\beta) = x_0^\top \operatorname{Cov}(\hat\beta)\, x_0
= \sigma^2\, x_0^\top (X^\top X)^{-1} x_0 .$$

Every term carries a factor $\sigma^2/n$, so this **shrinks as $1/\sqrt n$** and goes to zero with
enough data. It is the uncertainty in the *mean* relation.

**What will the next station read?** A new observation is $y_0 = x_0^\top\beta + \varepsilon_0$, and
$\varepsilon_0$ is a fresh draw, independent of everything used to fit. So

$$\operatorname{Var}(y_0 - x_0^\top\hat\beta)
= \underbrace{\sigma^2 x_0^\top (X^\top X)^{-1} x_0}_{\text{where the curve is}}
+ \underbrace{\sigma^2}_{\text{scatter of one reading}} .$$

The second term does not contain $n$. **No amount of data shrinks it**, because it is not ignorance
about the model — it is the spread of the thing being predicted.

Confusing the two is the commonest error in applied regression, and it is not a subtlety: below,
the two intervals differ by more than an order of magnitude on the same fit at the same point.

In [ ]:
x0 = np.array([1.0, 3.0, np.log10(50.0), 50.0])      # M 3.0 at R 50 km
var_mean = x0 @ Cov @ x0

ci = 1.96 * np.sqrt(var_mean)
pi = 1.96 * np.sqrt(var_mean + sigma_hat**2)

print(f"at M 3.0, R 50 km, fitted log10 A = {x0 @ beta_hat:.4f}")
print(f"  95% confidence interval on the curve      +/- {ci:.4f}")
print(f"  95% prediction interval for one reading   +/- {pi:.4f}")
print(f"  ratio {pi / ci:.1f}x")

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.0))
rr = np.logspace(np.log10(5), np.log10(300), 300)
Xs = np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0), np.log10(rr), rr])
mu = Xs @ beta_hat
vm = np.einsum("ij,jk,ik->i", Xs, Cov, Xs)

keep = np.abs(M - 3.0) < 0.25
ax.scatter(R[keep], y[keep], s=8, color="0.55", label="readings, M 3.0 +/- 0.25")
ax.fill_between(rr, mu - 1.96 * np.sqrt(vm + sigma_hat**2), mu + 1.96 * np.sqrt(vm + sigma_hat**2),
                color="C1", alpha=0.22, label="95% prediction interval")
ax.fill_between(rr, mu - 1.96 * np.sqrt(vm), mu + 1.96 * np.sqrt(vm),
                color="C0", alpha=0.85, label="95% confidence interval")
ax.plot(rr, mu, "k-", lw=1.2, label="fitted curve at M 3.0")
ax.set_xscale("log"); ax.set_xlabel("R (km)"); ax.set_ylabel(r"$\log_{10} A$")
ax.set_title("the same fit, two intervals"); ax.legend(fontsize=7, loc="lower left")
fig.tight_layout(); plt.show()

In [ ]:
# How each interval behaves as the network grows.
sizes = np.array([25, 50, 100, 250, 500, 1000, 2000, 5000, 10000])
rows = []
for m_ in sizes:
    r_ = np.random.default_rng(9000 + m_)
    Mm = r_.uniform(1.0, 4.5, m_); Rm = 10 ** r_.uniform(np.log10(5), np.log10(300), m_)
    Xm = np.column_stack([np.ones(m_), Mm, np.log10(Rm), Rm])
    ym = Xm @ BETA_TRUE + r_.normal(0, SIGMA_TRUE, m_)
    bm, *_ = np.linalg.lstsq(Xm, ym, rcond=None)
    rm = ym - Xm @ bm; sm2 = rm @ rm / (m_ - p)
    vm_ = x0 @ (sm2 * np.linalg.inv(Xm.T @ Xm)) @ x0
    rows.append((m_, 1.96 * np.sqrt(vm_), 1.96 * np.sqrt(vm_ + sm2)))

print(f"{'n':>7s} {'CI half-width':>15s} {'PI half-width':>15s}")
for m_, c_, pp in rows:
    print(f"{m_:7d} {c_:15.4f} {pp:15.4f}")

In [ ]:
arr = np.array(rows)
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.loglog(arr[:, 0], arr[:, 1], "o-", label="confidence interval")
ax.loglog(arr[:, 0], arr[:, 2], "s-", label="prediction interval")
ax.loglog(arr[:, 0], arr[0, 1] * np.sqrt(arr[0, 0] / arr[:, 0]), "k--", lw=0.8,
          label=r"$1/\sqrt{n}$")
ax.axhline(1.96 * SIGMA_TRUE, color="0.5", lw=0.8, ls=":")
ax.text(30, 1.96 * SIGMA_TRUE * 1.06, r"1.96 $\sigma$ — the floor", fontsize=7, color="0.4")
ax.set_xlabel("number of readings"); ax.set_ylabel("half-width at M 3.0, R 50 km")
ax.legend(fontsize=7); ax.set_title("one shrinks, one does not")
fig.tight_layout(); plt.show()

**Exercise 2.** The previous exercise showed the confidence interval covers *the truth*
about 95 % of the time. Now ask what it does to the question people actually use it for.

Over 2000 trials, draw a fresh reading $y_0$ at $x_0$ (M 3.0, R 50 km), refit on new data, and
count how often that reading falls inside the confidence band and how often inside the prediction
band. One of the two answers is 95 %. Predict which, and by roughly how much the other misses,
before you run it.

In [ ]:
# your code here


In [ ]:
# ── Checkpoint 2 ── run this if you are behind or something broke ──
# Nothing after A.5 depends on the exercises; this restates the two intervals at x0.
x0 = np.array([1.0, 3.0, np.log10(50.0), 50.0])
var_mean = x0 @ Cov @ x0
print(f"CI +/- {1.96*np.sqrt(var_mean):.4f}   PI +/- {1.96*np.sqrt(var_mean + sigma_hat**2):.4f}")

### A.6 · Collinearity: a good fit with meaningless coefficients

$\hat\beta$ came back close to the truth and every standard error was small. That is a property of
this design, not of least squares, and the two distance columns are about to show why.

$\log_{10}R$ and $R$ are different functions, but over a bounded range of $R$ they move together.
When two columns of $X$ are nearly proportional, $X^\top X$ is nearly singular, and
$(X^\top X)^{-1}$ — which is the covariance — has large entries in exactly those directions. The
consequence is specific and worth stating precisely:

> The **combination** $c_2\log_{10}R + c_3R$ is determined by the data. The **individual**
> $c_2$ and $c_3$ are not.

A published coefficient can therefore disagree with yours by many standard errors while both curves
pass through the same points, and reporting either coefficient alone, with its own error bar, hides
that the two are being traded against each other.

In [ ]:
corr_cols = np.corrcoef(np.log10(R), R)[0, 1]
D = np.sqrt(np.outer(np.diag(Cov), np.diag(Cov)))
corr_coef = (Cov / D)[2, 3]

print(f"correlation of the two COLUMNS,      log10 R vs R : {corr_cols:+.4f}")
print(f"correlation of the two COEFFICIENTS, c2 vs c3     : {corr_coef:+.4f}")
print(f"condition number of X                             : {np.linalg.cond(X):.1f}")
print("\nsingular values of X:", np.array2string(np.linalg.svd(X, compute_uv=False), precision=3))

In [ ]:
# Walk to the two ends of the long axis of the 95% ellipse for (c2, c3) and ask
# whether the predicted curves can be told apart.
sub = Cov[2:, 2:]
w, V = np.linalg.eigh(sub)
axis = V[:, -1] * np.sqrt(w[-1] * stats.chi2.ppf(0.95, 2))
end_a, end_b = beta_hat[2:] + axis, beta_hat[2:] - axis

curves = []
for end in (end_a, end_b):
    b_alt = np.r_[beta_hat[:2], end]
    curves.append(np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0), np.log10(rr), rr]) @ b_alt)

print(f"corner A of the ellipse: c2 = {end_a[0]:+.4f}, c3 = {end_a[1]:+.6f}")
print(f"corner B of the ellipse: c2 = {end_b[0]:+.4f}, c3 = {end_b[1]:+.6f}")
gap = np.abs(curves[0] - curves[1]).max()
print(f"c2 differs between the corners by {abs(end_a[0]-end_b[0]) / se[2]:.1f} standard errors,")
print(f"yet the two predicted curves differ by at most {gap:.4f} log10 units -- which is "
      f"{gap / (1.96*np.sqrt(var_mean + sigma_hat**2)):.2f} of the")
print("half-width of the prediction interval for a single reading, from A.5.")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.5))
th = np.linspace(0, 2 * np.pi, 200)
ell = (V @ np.diag(np.sqrt(w * stats.chi2.ppf(0.95, 2))) @ np.vstack([np.cos(th), np.sin(th)])).T
a1.plot(beta_hat[2] + ell[:, 0], beta_hat[3] + ell[:, 1], "C0-", lw=1.2)
a1.plot(*beta_hat[2:], "C0+", ms=9, label="fit")
a1.plot(*BETA_TRUE[2:], "kx", ms=8, label="truth")
a1.plot([end_a[0], end_b[0]], [end_a[1], end_b[1]], "C3o", ms=5, label="ellipse ends")
a1.set_xlabel(r"$c_2$"); a1.set_ylabel(r"$c_3$"); a1.legend(fontsize=7)
a1.set_title("95% joint region: a ridge, not a blob")

mu_ = np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0), np.log10(rr), rr]) @ beta_hat
vm_ = np.einsum("ij,jk,ik->i", np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0),
                                                np.log10(rr), rr]), Cov,
                np.column_stack([np.ones_like(rr), np.full_like(rr, 3.0), np.log10(rr), rr]))
a2.scatter(R[np.abs(M - 3.0) < 0.25], y[np.abs(M - 3.0) < 0.25], s=6, color="0.75", zorder=0)
a2.fill_between(rr, mu_ - 1.96 * np.sqrt(vm_ + sigma_hat**2),
                mu_ + 1.96 * np.sqrt(vm_ + sigma_hat**2), color="C1", alpha=0.18, zorder=1,
                label="95% prediction interval")
a2.plot(rr, curves[0], "C3-", lw=1.4, zorder=3, label="corner A")
a2.plot(rr, curves[1], "C3--", lw=1.4, zorder=3, label="corner B")
a2.set_xscale("log"); a2.set_xlabel("R (km)"); a2.set_ylabel(r"$\log_{10} A$ at M 3.0")
a2.legend(fontsize=7, loc="lower left")
a2.set_title("both are consistent with these data")
fig.tight_layout(); plt.show()

It is tempting to look for a reparameterisation that makes the problem go away. Hutton &
Boore write their distance terms centred on a reference distance, $\log_{10}(R/100)$ and $R-100$,
and centring is the standard advice for collinearity. Test it.

In [ ]:
Xc = np.column_stack([np.ones(n), M, np.log10(R / 100.0), R - 100.0])
bc, *_ = np.linalg.lstsq(Xc, y, rcond=None)
rc = y - Xc @ bc
Cc = (rc @ rc / (n - p)) * np.linalg.inv(Xc.T @ Xc)
Dc = np.sqrt(np.outer(np.diag(Cc), np.diag(Cc)))

print(f"uncentred: corr(c2, c3) = {corr_coef:+.4f}, cond(X) = {np.linalg.cond(X):8.1f}")
print(f"centred  : corr(c2, c3) = {(Cc/Dc)[2, 3]:+.4f}, cond(X) = {np.linalg.cond(Xc):8.1f}")
print(f"\nresidual sum of squares, uncentred {resid @ resid:.4f} vs centred {rc @ rc:.4f}"
      "   -- the same fit in different coordinates")

The conditioning improves and the intercept becomes interpretable — it is now the value at
the reference distance rather than an extrapolation to $R=1$ km. **The correlation between the two
slopes does not move at all.** Shifting a column by a constant changes how that column relates to
the intercept; it cannot change how two slopes trade against each other. Centring is worth doing,
and it is not a cure.

What is actually identified is a *direction*. The eigenvectors of the $(c_2,c_3)$ covariance are the
long and short axes of the ellipse: one combination the data pin down, one they do not.

In [ ]:
# Do this on the CORRELATION matrix, not the covariance. c3 is a thousand times smaller
# than c2, so eigenvectors of the raw covariance report the choice of units, not the
# identifiability. Measuring each coefficient in its own standard errors removes that.
corr2 = np.array([[1.0, corr_coef], [corr_coef, 1.0]])
wc, Vc = np.linalg.eigh(corr2)
print(f"well-determined direction: {Vc[0, 0]:+.3f} c2/se2 {Vc[1, 0]:+.3f} c3/se3,"
      f"  sd {np.sqrt(wc[0]):.3f}")
print(f"poorly determined:         {Vc[0, 1]:+.3f} c2/se2 {Vc[1, 1]:+.3f} c3/se3,"
      f"  sd {np.sqrt(wc[1]):.3f}")
print(f"the data constrain one direction {np.sqrt(wc[1] / wc[0]):.1f}x better than the other")
print("\nThe well-determined direction moves the two coefficients TOGETHER; the poorly")
print("determined one trades them against each other, which is what a negative correlation")
print("means. Report the combination, or drop a term -- c2 alone, with its own error bar,")
print("states a precision the data never had.")

In [ ]:
# ── Checkpoint 3 ── run this if you are behind or something broke ──
# Everything A.7 needs.
Cov = sigma_hat**2 * np.linalg.inv(XtX)
se = np.sqrt(np.diag(Cov))
sv = np.linalg.svd(X, compute_uv=False)
print("singular values:", np.round(sv, 3))

### A.7 · Ridge regression is MAP under a Gaussian prior

The trade-off in A.6 is the data declining to separate two directions. One response is to say
something, before looking at the data, about how large the coefficients ought to be. Doing that
carefully turns out to give a familiar formula.

1. **The forward model, probabilistically.** $y = X\beta + \varepsilon$ with
   $\varepsilon\sim N(0,\sigma^2 I)$, so
   $p(y\mid\beta) \propto \exp\!\big[-\lVert y-X\beta\rVert^2/2\sigma^2\big]$ — the A.3 likelihood.
2. **The prior.** State it before the data: $\beta \sim N(0, s^2 I)$, so
   $p(\beta) \propto \exp\!\big[-\lVert\beta\rVert^2/2s^2\big]$. Here $s$ is a real quantity in the
   units of $\beta$ — the size of coefficient we would be unsurprised by.
3. **Bayes.** $p(\beta\mid y) \propto p(y\mid\beta)\,p(\beta)$. The denominator does not contain
   $\beta$, so it cannot move the maximum and is dropped.
4. **Negative log.** The product becomes a sum of two quadratics:
   $-\log p(\beta\mid y) = \dfrac{\lVert y - X\beta\rVert^2}{2\sigma^2} + \dfrac{\lVert\beta\rVert^2}{2s^2} + \text{const}$.
5. **Rescale** by $2\sigma^2$, which is positive and so does not move the minimiser:
   $\lVert y-X\beta\rVert^2 + \lambda\lVert\beta\rVert^2$ with
   $\boxed{\lambda = \sigma^2/s^2}$. This is the damped objective, and $\lambda$ is now *read off*
   rather than tuned: it is the ratio of how noisy the data are to how large the coefficients were
   expected to be.
6. **Minimise.** Differentiating gives $(X^\top X + \lambda I)\hat\beta_\lambda = X^\top y$. Adding
   $\lambda I$ raises every eigenvalue of $X^\top X$ by $\lambda$, so for $\lambda>0$ the matrix is
   positive definite and invertible **even when $X^\top X$ is singular** — which is what makes this
   the standard move for an under-determined inverse problem, in week 12 and again in week 14.
7. **The mechanism.** Write $X = UDV^\top$. Then
   $\hat\beta_\lambda = \sum_i f_i \dfrac{u_i^\top y}{d_i} v_i$ with **filter factors**
   $f_i = d_i^2/(d_i^2+\lambda)$. Directions the data constrain well ($d_i^2 \gg \lambda$) pass
   through untouched; directions the data barely see are damped toward zero. Ridge does not shrink
   $\beta$ uniformly — it shrinks *the directions the data are quiet about*.

In practice the columns are standardised first and the intercept is left unpenalised, so that the
prior means the same thing regardless of the units each column happens to be in.

In [ ]:
# Filter factors on this design, at five values of lambda.
print(f"{'lambda':>10s}  " + "  ".join(f"f{i+1}" for i in range(p)))
for lam in [0.0, 1e0, 1e2, 1e4, 1e6]:
    f = sv**2 / (sv**2 + lam)
    print(f"{lam:10.0e}  " + "  ".join(f"{v:.4f}" for v in f))
print("\nthe best-resolved direction is untouched at every lambda; the weakest is damped first")

Filter factors describe what ridge *does*; they do not say when it is worth doing. On the
four-column design it is not: those four parameters are estimated from 2000 readings, every
singular value is comfortably above zero, and a prior displaces a well-determined fit for nothing.

The prior earns its place when the model has many parameters that each see little data. The
canonical case in this problem is a **site term**: one coefficient per station, absorbing the fact
that a station on soft sediment reads systematically higher than one on rock. Add sixty of them and
the design has sixty-four columns, four readings per station — and each site term is estimated from
those four readings alone.

Now $\lambda$ stops being a knob. Step 5 of the derivation gives $\lambda = \sigma^2/s^2$, and both
quantities are measurable: $\sigma$ is the scatter of repeat readings at one station, $s$ the
spread of the site terms across stations. The values below are the ones measured on the real
southern-California network, and Part B measures them again.

In [ ]:
SIGMA_W, TAU_SITE, N_STA = 0.239, 0.245, 60   # within-station, between-station, stations
SITE_LAM = SIGMA_W**2 / TAU_SITE**2

def site_design(r_, m_):
    """The A.1 design, plus one indicator column per station."""
    Mm = r_.uniform(1.0, 4.5, m_)
    Rm = 10 ** r_.uniform(np.log10(5), np.log10(300), m_)
    which = r_.integers(0, N_STA, m_)
    D = np.zeros((m_, N_STA)); D[np.arange(m_), which] = 1.0
    return np.column_stack([np.ones(m_), Mm, np.log10(Rm), Rm, D])

def site_ridge(Xm, ym, lam):
    """Penalise the site columns only: the prior is about stations, not about attenuation."""
    pen = np.zeros(Xm.shape[1]); pen[4:] = lam
    return np.linalg.solve(Xm.T @ Xm + np.diag(pen), Xm.T @ ym)

_probe = site_design(np.random.default_rng(0), 250)
_empty = int((_probe[:, 4:].sum(axis=0) == 0).sum())
print(f"design: {_probe.shape[1]} columns, rank {np.linalg.matrix_rank(_probe)}")
print("  -1  the intercept column is the sum of the sixty site columns")
print(f"  -{_empty}  station(s) drew no readings at all, so their column is identically zero")
print("X'X is therefore SINGULAR: only DIFFERENCES between site terms are identifiable, and a")
print("station with no data has no estimate. Step 6 of the derivation is not hypothetical here")
print("-- for lambda > 0 the matrix is invertible, which is the only reason a fit exists.\n")
print(f"within-station scatter  sigma = {SIGMA_W}")
print(f"between-station scatter s     = {TAU_SITE}")
print(f"so the derivation predicts lambda = sigma^2 / s^2 = {SITE_LAM:.2f}")

In [ ]:
LAMS = np.logspace(-2, 3, 26)

def held_out(m_, trials=200, ntest=800):
    """Mean squared error on fresh readings from the same stations."""
    err = np.zeros(len(LAMS))
    for t in range(trials):
        r_ = np.random.default_rng(7000 + t)
        site = r_.normal(0, TAU_SITE, N_STA)
        full = np.r_[BETA_TRUE, site]
        Xtr = site_design(r_, m_); ytr = Xtr @ full + r_.normal(0, SIGMA_W, m_)
        Xte = site_design(r_, ntest); yte = Xte @ full + r_.normal(0, SIGMA_W, ntest)
        for i, lam in enumerate(LAMS):
            err[i] += np.mean((yte - Xte @ site_ridge(Xtr, ytr, lam)) ** 2)
    return err / trials

curves_site = {m_: held_out(m_) for m_ in (250, 600, 3000)}
for m_, c in curves_site.items():
    j = int(c.argmin())
    print(f"n = {m_:5d} ({m_ / N_STA:4.1f} readings/station): best lambda {LAMS[j]:6.3f}, "
          f"held-out error {c[j]:.4f} against {c[0]:.4f} at the weakest prior tried "
          f"({100 * (c[0] - c[j]) / c[0]:+.1f}%)")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.4))
for m_, c in curves_site.items():
    a1.semilogx(LAMS, c, "o-", ms=3, label=f"n = {m_}")
    a1.plot(LAMS[c.argmin()], c.min(), "k*", ms=11)
a1.axvline(SITE_LAM, color="C3", ls="--", lw=1.0)
a1.text(SITE_LAM * 1.15, a1.get_ylim()[1] * 0.97, r"$\sigma^2/s^2$", color="C3", fontsize=8,
        va="top")
a1.set_xlabel(r"$\lambda$"); a1.set_ylabel("held-out mean squared error")
a1.legend(fontsize=7); a1.set_title("the prior pays when each station is thin")

_r = np.random.default_rng(7000)
site_true = _r.normal(0, TAU_SITE, N_STA)
Xs_ = site_design(_r, 250); ys_ = Xs_ @ np.r_[BETA_TRUE, site_true] + _r.normal(0, SIGMA_W, 250)
for lam, col in ((1e-2, "0.6"), (SITE_LAM, "C0")):
    est = site_ridge(Xs_, ys_, lam)[4:]
    a2.plot(site_true - site_true.mean(), est - est.mean(), "o", ms=4, color=col,
            label=f"lambda = {lam:.2f}")
a2.plot([-0.7, 0.7], [-0.7, 0.7], "k-", lw=0.8)
a2.set_xlabel("true site term (centred)"); a2.set_ylabel("estimated site term (centred)")
a2.legend(fontsize=7); a2.set_title("shrinkage, station by station")
fig.tight_layout(); plt.show()

**Exercise 3.** The curve on the left cannot be drawn without knowing the true site terms,
so it is not available on real data. Cross-validation is: hold out part of the data, fit on the
rest, score the prediction on what was held out — and it never looks at the truth.

Build one 250-reading network, run 5-fold cross-validation over `LAMS`, and report the $\lambda$ it
selects. Compare it with $\sigma^2/s^2$ from the derivation. Agreement here is the claim of step 5
being confirmed by a procedure that knows nothing about it.

In [ ]:
# your code here


### A.8 · When the noise is not Gaussian

Return to the recipe in A.3: **(i)** write the likelihood, **(ii)** maximise it, **(iii)** invert
the curvature at the maximum. Steps (i) and (iii) are general. Step (ii) collapsed into the normal
equations only because the log-likelihood was a sum of squares — and it was a sum of squares only
because the noise was Gaussian.

Take a quantity $x \ge x_{\min}$ that is exponentially distributed,
$p(x) = \beta e^{-\beta(x - x_{\min})}$. Two ways to estimate $\beta$ from a sample:

**Least squares on a histogram.** Bin the sample, count how many observations exceed each bin edge,
take $\log_{10}$ of the counts, and fit a straight line. The slope gives $\beta$. This is a
regression, so all of A.2–A.5 applies — except that its assumptions do not: the counts are Poisson,
so their scatter grows with the count rather than being constant; and each cumulative count contains
every count above it, so the points are strongly correlated rather than independent. Both violations
are invisible in the plot, which looks like an excellent straight line.

**Maximum likelihood.** Follow the recipe. The log-likelihood is
$\ell(\beta) = n\log\beta - \beta\sum_i (x_i - x_{\min})$; setting $\partial\ell/\partial\beta = 0$
gives $\hat\beta = 1/(\bar x - x_{\min})$, and $-\partial^2\ell/\partial\beta^2 = n/\beta^2$, so by
step (iii) the standard error is $\hat\beta/\sqrt{n}$. No design matrix, no regression, and an error
bar that came from the same place as the one in A.4.

Both are computed below on the same samples, and — because the data are generated — both can be
judged against the truth and against how much they actually vary from sample to sample.

In [ ]:
BETA_EXP = np.log(10)      # rate of the exponential; the truth to recover
X_MIN, BIN = 0.0, 0.1      # values reported rounded to the nearest 0.1
N_SAMPLE = 1000

def one_sample(seed):
    r_ = np.random.default_rng(seed)
    x = (X_MIN - BIN / 2) + r_.exponential(1 / BETA_EXP, N_SAMPLE)
    xb = np.round(x / BIN) * BIN                        # what the catalogue would report
    mle = 1.0 / (xb.mean() - X_MIN + BIN / 2)           # rounding shifts the effective minimum
    edges = np.round(np.arange(X_MIN, xb.max() + BIN, BIN), 4)
    cnt = np.array([(xb >= e - 1e-9).sum() for e in edges])
    k = cnt > 0
    A = np.column_stack([np.ones(k.sum()), edges[k]])
    g, *_ = np.linalg.lstsq(A, np.log10(cnt[k]), rcond=None)
    rr_ = np.log10(cnt[k]) - A @ g
    se_ls = np.sqrt(np.diag((rr_ @ rr_ / (k.sum() - 2)) * np.linalg.inv(A.T @ A)))[1]
    return mle, mle / np.sqrt(N_SAMPLE), -g[1] * np.log(10), se_ls * np.log(10)

print("one sample:", np.round(one_sample(0), 4))

In [ ]:
draws = np.array([one_sample(s) for s in range(400)])
mle, mle_se, lsq, lsq_se = draws.T

print(f"truth                                       {BETA_EXP:8.4f}")
print(f"maximum likelihood, mean over 400 samples   {mle.mean():8.4f}")
print(f"least squares,      mean over 400 samples   {lsq.mean():8.4f}"
      f"   ({100 * (lsq.mean() - BETA_EXP) / BETA_EXP:+.1f}% biased)")
print()
print(f"actual spread of the MLE           {mle.std():8.4f}   it reports {mle_se.mean():8.4f}")
print(f"actual spread of least squares     {lsq.std():8.4f}   it reports {lsq_se.mean():8.4f}")
print()
print(f"least squares is {lsq.std() / mle.std():.1f}x more variable than the MLE,")
print(f"and understates its own spread by a factor of {lsq.std() / lsq_se.mean():.1f}")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.4, 3.4))
r_ = np.random.default_rng(4)
x = (X_MIN - BIN / 2) + r_.exponential(1 / BETA_EXP, N_SAMPLE)
xb = np.round(x / BIN) * BIN
edges = np.round(np.arange(X_MIN, xb.max() + BIN, BIN), 4)
cnt = np.array([(xb >= e - 1e-9).sum() for e in edges])
k = cnt > 0
a1.semilogy(edges[k], cnt[k], "o", ms=3.5, color="0.3")
gg, *_ = np.linalg.lstsq(np.column_stack([np.ones(k.sum()), edges[k]]), np.log10(cnt[k]), rcond=None)
a1.semilogy(edges[k], 10 ** (gg[0] + gg[1] * edges[k]), "C3-", lw=1.2, label="least squares")
a1.set_xlabel("x"); a1.set_ylabel("count of values above x")
a1.set_title("the fit that looks unimpeachable"); a1.legend(fontsize=7)

bins = np.linspace(min(lsq.min(), mle.min()), max(lsq.max(), mle.max()), 45)
a2.hist(lsq, bins=bins, alpha=0.6, label="least squares", color="C3")
a2.hist(mle, bins=bins, alpha=0.6, label="maximum likelihood", color="C0")
a2.axvline(BETA_EXP, color="k", lw=1.2, label="truth")
a2.set_xlabel(r"estimated $\beta$"); a2.set_ylabel("samples")
a2.set_title("400 samples, same data, two estimators"); a2.legend(fontsize=7)
fig.tight_layout(); plt.show()

The straight line on the left is not a bad fit — it is a very good fit, to the wrong
objective. Least squares here is still minimising a sum of squares; it is just no longer the
maximum-likelihood estimator, because the noise it implicitly assumes is not the noise the data
have. The price is paid three times over: the estimate is biased, it varies more from sample to
sample than it needs to, and the standard error it reports is the one belonging to the model it
assumed rather than the one that generated the data.

The third of those is the dangerous one. A biased estimate with an honest error bar advertises its
own trouble. This one does not.

Part C is this section with $x$ replaced by earthquake magnitude.

## Part B · Determining a magnitude

Part A worked where the answer was known. Here it is not, and the question is the one the course
opened with: **an earthquake happened, stations recorded it — what is its magnitude, and how well
do we know that?**

Every result from Part A is used, and each section says which:

| from Part A | used in |
|---|---|
| A.1 a model is a design matrix | B.2 — and one coefficient turns out to be a *definition*, not a fit |
| A.2 normal equations · A.4 the covariance | B.3 |
| A.6 collinearity | B.3 — worse here than in Part A, and for a reason worth seeing |
| A.3 least squares is MLE **under Gaussian noise** | B.4 — where two stations are not Gaussian draws |
| A.5 confidence versus prediction interval | B.5 — this *is* the answer to "how well do we know it" |
| A.7 step 6, a singular design | B.6 — the parameter this dataset cannot estimate |

**On scale.** This is one earthquake at 33 stations. A published local-magnitude study uses of order
$10^5$ readings from $10^4$ events. Nothing in the procedure below changes at that scale — the same
design matrix, the same estimator, the same intervals. What changes is *which parameters are
identifiable*, and B.6 shows precisely which one falls off the end of a small dataset and why. That
is worth knowing directly rather than being told.

### B.1 · The measurement

Section 7 of the project notebook took the reference event — 2016-03-31, The Geysers, catalogue
magnitude 2.91 — removed each instrument's response, simulated the Wood–Anderson torsion
seismometer from its poles and zeros, and measured the peak of the result. That is the amplitude a
magnitude is defined on, and this section starts from it.

**With one change.** Section 7 measured on the *vertical* component, because it was demonstrating
response removal and the vertical is where a P arrival is clearest. A local magnitude is defined on
the **horizontal** — Richter's instrument was a horizontal torsion seismometer, and the amplitude
entering the scale is a horizontal one. So the measurement is repeated here on both horizontal
components of each station, and B.7 measures what that change is worth.

The measurement itself is section 7's, and its result is carried here rather than refetched:
this notebook makes **no network call at all**, so a room of eight people on one connection is not
a way for the session to fail. Both component sets are carried, because B.7 needs the comparison.

In [ ]:
WOOD_ANDERSON = {"poles": [-6.283 - 4.712j, -6.283 + 4.712j], "zeros": [0j],
                 "gain": 1.0, "sensitivity": 2080.0}

def deconvolution_band(sps):
    """Section 7's rule: the taper must sit below Nyquist for THIS trace.

    A single fixed band -- (0.2, 0.5, 40, 50) Hz is conventional -- is above Nyquist for every
    channel sampled below 100 sps, and using it everywhere makes short-period stations read three
    orders of magnitude low. The error then looks exactly like a site effect.
    """
    return (0.2, 0.5, min(40.0, 0.35 * sps), min(50.0, 0.45 * sps))

def saturated(tr, rail=2048, margin=0.90):
    """A trace merely NEAR the 12-bit rail has already had its peak flattened."""
    a = np.abs(np.asarray(tr.data, dtype=float))
    peak = a.max() if a.size else 0.0
    return bool(peak) and peak <= rail * 1.03 and peak >= rail * margin

print("Wood-Anderson: 2 poles, 1 zero, static magnification 2080")

In [ ]:
HORIZONTAL = """network,station,band,sps,epi_km,hyp_km,amp_mean_mm,amp_max_mm
BG,NEG,DP,500.0,1.003,2.3,32.314095,32.361794
BG,ACR,DP,500.0,1.503,2.558,7.841474,9.039727
BG,LCK,DP,500.0,1.875,2.793,21.978147,28.762819
BG,STY,DP,500.0,1.91,2.816,9.025966,13.20376
BG,HVC,DP,500.0,2.444,3.203,21.178744,27.503039
BG,CLV,DP,500.0,2.914,3.575,28.152087,41.06975
NC,GDXB,HH,100.0,3.291,3.888,18.57338,19.111861
BG,FUM,DP,500.0,3.995,4.5,15.635207,31.233173
BG,SQK,DP,500.0,4.093,4.586,12.737505,15.173338
BG,BRP,DP,500.0,4.596,5.041,10.171686,11.1375
BG,TCH,DP,500.0,4.999,5.41,8.345529,10.25314
BG,DRK,DP,500.0,5.225,5.62,10.509144,11.858144
BG,MNS,DP,500.0,5.302,5.692,34.157148,35.791423
BG,FNF,DP,500.0,5.855,6.21,2.724386,3.919188
BG,SB4,DP,500.0,5.968,6.317,11.511872,14.998947
BG,BUC,DP,500.0,6.255,6.588,6.935946,9.539469
BG,MCL,DP,500.0,6.295,6.626,4.929385,5.286497
BG,AL4,DP,500.0,6.519,6.84,11.849454,12.117958
BG,RGP,DP,500.0,7.417,7.7,5.923436,11.837492
NP,ADS2,HN,200.0,7.707,7.98,2.45936,2.746293
BG,PFR,DP,500.0,8.009,8.272,0.005419,0.005736
BG,AL3,DP,500.0,8.16,8.419,9.538772,11.495452
BG,DES,DP,500.0,8.241,8.497,4.674387,5.587244
BG,AL6,DP,500.0,8.921,9.158,5.931043,7.493659
BG,AL5,DP,500.0,9.35,9.577,7.071546,7.754115
BG,HBW,DP,500.0,10.531,10.733,5.331543,5.814308
BG,AL1,DP,500.0,10.58,10.781,2.707733,5.411247
BG,AL2,DP,500.0,11.738,11.919,2.847208,3.512724
BG,HER,DP,500.0,13.294,13.454,4.503848,5.300557
NC,NHS,EH,100.0,22.568,22.663,2.510074,3.611384
BK,MNRC,HH,100.0,28.403,28.478,1.003486,1.031925
NC,N005,HN,200.0,32.239,32.306,0.367822,0.415605
BK,HOPS,HH,100.0,32.816,32.881,0.714456,0.845798"""

VERTICAL = """station,hyp_km,amp_mm
NEG,2.3,9.17779
ACR,2.558,4.465688
LCK,2.793,9.325485
STY,2.816,4.932012
HVC,3.203,4.532467
CLV,3.575,0.732203
GDXB,3.888,3.713943
FUM,4.5,7.935823
SQK,4.586,7.127171
BRP,5.041,6.281376
TCH,5.41,3.714683
DRK,5.62,2.405089
MNS,5.692,7.346048
FNF,6.21,1.301032
SB4,6.317,4.577267
BUC,6.588,2.131039
MCL,6.626,2.271848
AL4,6.84,5.3796
RGP,7.7,3.449504
ADS2,7.98,1.427638
PFR,8.272,0.004437
AL3,8.419,2.768795
DES,8.497,0.948364
AL6,9.158,2.300478
AL5,9.577,1.680268
HBW,10.733,2.293233
AL1,10.781,2.366251
AL2,11.919,1.247685
GAXB,12.699,1.645459
HER,13.454,0.003036
NHS,22.663,1.011135
GCVB,22.741,0.87331
MNRC,28.478,0.670253
NMW,30.616,0.273449
N005,32.306,0.171795
HOPS,32.881,0.580309
NPV,35.401,0.352256"""

wa = pd.read_csv(io.StringIO(HORIZONTAL)).sort_values("hyp_km").reset_index(drop=True)
vert = pd.read_csv(io.StringIO(VERTICAL)).sort_values("hyp_km").reset_index(drop=True)
CATALOGUE_MAG, CATALOGUE_TYPE = 2.91, "d (coda duration)"

print(f"{len(wa)} stations with both horizontals, "
      f"{wa.hyp_km.min():.1f} to {wa.hyp_km.max():.1f} km from the source")
print(f"{len(vert)} vertical-component readings, for the comparison in B.7")
print(f"duplicate station rows: {int(wa.station.duplicated().sum())}")
print(f"catalogue magnitude for this event: {CATALOGUE_MAG} ({CATALOGUE_TYPE})")
print()
print(wa.head(4).to_string(index=False))

### B.2 · A magnitude is an equation

Richter's definition is one line. The magnitude of an event, measured at a station that recorded a
Wood–Anderson amplitude $A$ (in mm) at distance $R$, is

$$M_L = \log_{10} A \;+\; \big(-\log_{10} A_0(R)\big),$$

where $-\log_{10}A_0$ is a **distance correction**: how much a given amplitude should be scaled up
to account for the wave having spread and been absorbed on its way to the station. All of the
physics is in that second term, and it is the term that has to be measured from data.

Now read it as a design matrix, as A.1 taught. Writing the correction in the standard parametric
form used since Richter,

$$\log_{10} A = c_0 + c_1 M + c_2 \log_{10} R + c_3 R,$$

this is the model of Part A exactly — the same four columns. But one thing has changed, and it is
worth stopping on, because Part A trained the instinct to fit every coefficient.

**$c_1 = 1$ is not fitted. It is the definition.** A magnitude is *defined* as a logarithm of
amplitude, so a unit increase in $M$ means a factor of ten in $A$ by construction. Estimating $c_1$
from data would not be testing the definition; it would be testing whether the *catalogue's*
magnitudes — which for this event is a coda-duration measurement, an entirely different procedure —
happen to scale with amplitude the way the local-magnitude scale does. That is a real question, and
it is not this one.

So with $c_1$ fixed at 1 and one event, what remains to estimate is the distance correction:
$c_0$, $c_2$ and $c_3$.

### B.3 · Fitting the distance correction, and what A.6 predicted

Three parameters, 33 readings. Part A's machinery applies unchanged — the design matrix, the normal
equations, $\hat\sigma^2 = \mathrm{RSS}/(n-p)$, and $\mathrm{Cov} = \sigma^2(X^\top X)^{-1}$.

A.6 makes a prediction before the fit is run. $\log_{10}R$ and $R$ were strongly correlated in Part
A at $n = 2000$; here $n$ is smaller by a factor of sixty and the distance range is narrower, so the
two columns should be *harder* to separate, not easier. Watch the coefficient correlation.

In [ ]:
X = np.column_stack([np.ones(len(wa)), np.log10(wa.hyp_km), wa.hyp_km])
y = np.log10(wa.amp_mean_mm.values)
p_b = X.shape[1]

beta_b, *_ = np.linalg.lstsq(X, y, rcond=None)
res_b = y - X @ beta_b
sig_b = np.sqrt(res_b @ res_b / (len(wa) - p_b))
Cov_b = sig_b**2 * np.linalg.inv(X.T @ X)
se_b = np.sqrt(np.diag(Cov_b))
D_b = np.sqrt(np.outer(np.diag(Cov_b), np.diag(Cov_b)))

for nm, v, s_ in zip(["c0", "c2 (log10 R)", "c3 (R)"], beta_b, se_b):
    print(f"{nm:>14s} = {v:+9.4f}  +/- {s_:.4f}   ({abs(v / s_):5.1f} standard errors from zero)")
print(f"\nsigma = {sig_b:.3f} log10 units,  n = {len(wa)}")
print(f"corr(c2, c3) = {(Cov_b / D_b)[1, 2]:+.4f}    against {-0.894:+.4f} in Part A at n = 2000")
print(f"cond(X)      = {np.linalg.cond(X):.1f}")

Read the standard errors before the coefficients. **Neither distance term is individually
resolved** — each sits close enough to zero that the data cannot exclude its absence — while the
correlation between them is even stronger than Part A's. This is A.6 with the volume turned up: the
*combination* of the two terms is what the 33 readings determine, and the split between geometrical
spreading and absorption is not.

That is not a defect in the data. It is what thirty-three readings over a limited distance range can
say, and the honest response is the one A.6 gave: report the combination, or reduce the model. What
a published study buys with $10^5$ readings over a wide distance range is exactly the ability to
separate these two, which is why those studies exist.

In [ ]:
# What IS determined: the curve, even where the coefficients are not.
rr_b = np.logspace(np.log10(wa.hyp_km.min()), np.log10(wa.hyp_km.max()), 200)
Xg = np.column_stack([np.ones_like(rr_b), np.log10(rr_b), rr_b])
mu_b = Xg @ beta_b
vm_b = np.einsum("ij,jk,ik->i", Xg, Cov_b, Xg)

fig, ax = plt.subplots(figsize=(6.2, 4.2))
ax.fill_between(rr_b, mu_b - 1.96 * np.sqrt(vm_b + sig_b**2),
                mu_b + 1.96 * np.sqrt(vm_b + sig_b**2), color="C1", alpha=0.18,
                label="95% prediction interval")
ax.fill_between(rr_b, mu_b - 1.96 * np.sqrt(vm_b), mu_b + 1.96 * np.sqrt(vm_b),
                color="C0", alpha=0.5, label="95% confidence interval")
ax.plot(rr_b, mu_b, "k-", lw=1.2)
ax.plot(wa.hyp_km, y, "o", ms=5, mfc="0.75", mec="0.3", mew=0.5, label="stations")
ax.set_xscale("log")
ax.set_xlabel("hypocentral distance (km)")
ax.set_ylabel(r"$\log_{10}$ Wood-Anderson amplitude (mm)")
ax.set_title(f"one earthquake, {len(wa)} stations")
ax.legend(fontsize=7)
fig.tight_layout(); plt.show()

In [ ]:
# ── Checkpoint 4 ── run this if you are behind or something broke ──
# Part B state: the amplitudes, the fit, and its covariance.
X = np.column_stack([np.ones(len(wa)), np.log10(wa.hyp_km), wa.hyp_km])
y = np.log10(wa.amp_mean_mm.values); p_b = X.shape[1]
beta_b, *_ = np.linalg.lstsq(X, y, rcond=None)
res_b = y - X @ beta_b
sig_b = np.sqrt(res_b @ res_b / (len(wa) - p_b))
Cov_b = sig_b**2 * np.linalg.inv(X.T @ X)
print("c0, c2, c3 =", np.round(beta_b, 4), " sigma =", round(sig_b, 3))

### B.4 · Two stations that are not Gaussian draws

A.3 showed that least squares is the maximum-likelihood estimator **when the errors are Gaussian**.
That was not a technicality to be waved past. It is an assumption about how far from the curve a
point is allowed to fall, and it can be checked.

Section 7.5 of the project notebook found two stations lying about three decades below the trend.
They are not clipped, contain no gaps, sit on a normal noise floor and recorded the event with good
signal-to-noise using the same instrument as their neighbours. The conclusion there was that the
fault is in the **bookkeeping rather than in the Earth** — a gain applied at the station, or written
into the response, that the metadata do not describe — and that it cannot be settled from inside a
notebook.

Whatever their cause, the question here is what they do to an estimator that assumes Gaussian
noise.

In [ ]:
z = res_b / sig_b
odd = wa.assign(resid=res_b, z=z).nsmallest(3, "resid")[["station", "hyp_km", "amp_mean_mm",
                                                         "resid", "z"]]
print(odd.to_string(index=False))
print()
for _, r_ in odd.iterrows():
    print(f"  {r_.station}: {abs(r_.z):5.1f} standard deviations below the curve. Under a Gaussian "
          f"likelihood the chance of a deviation that large is {2 * stats.norm.sf(abs(r_.z)):.2e}")

In [ ]:
# The same fit with and without them, and with a loss that does not assume Gaussian noise.
mask = res_b > -1.5
Xk, yk = X[mask], y[mask]
beta_keep, *_ = np.linalg.lstsq(Xk, yk, rcond=None)

def huber_fit(X_, y_, delta=1.0, iters=50):
    """Iteratively reweighted least squares: quadratic near zero, linear in the tail.

    Changing the loss IS changing the noise model -- a linear tail says large residuals are far more
    probable than a Gaussian allows, so a distant point pulls with bounded, not growing, force.
    """
    b = np.linalg.lstsq(X_, y_, rcond=None)[0]
    for _ in range(iters):
        r_ = y_ - X_ @ b
        s_ = 1.4826 * np.median(np.abs(r_ - np.median(r_))) or 1.0
        w_ = np.clip(delta / np.abs(r_ / s_), None, 1.0)
        b = np.linalg.lstsq(X_ * np.sqrt(w_)[:, None], y_ * np.sqrt(w_), rcond=None)[0]
    return b

beta_hub = huber_fit(X, y)
print(f"{'':>22s} {'c0':>9s} {'c2':>9s} {'c3':>10s}")
print(f"{'least squares, all ' + str(len(wa)):>22s} {beta_b[0]:9.3f} {beta_b[1]:9.3f} {beta_b[2]:10.5f}")
print(f"{'least squares, ' + str(int(mask.sum())) + ' kept':>22s} {beta_keep[0]:9.3f} "
      f"{beta_keep[1]:9.3f} {beta_keep[2]:10.5f}")
print(f"{'Huber, all ' + str(len(wa)):>22s} {beta_hub[0]:9.3f} {beta_hub[1]:9.3f} {beta_hub[2]:10.5f}")
r_keep = yk - Xk @ beta_keep
sig_keep = np.sqrt(r_keep @ r_keep / (mask.sum() - p_b))
print(f"\n{int((~mask).sum())} point of {len(wa)} removed:")
print(f"  c2                  {beta_b[1]:+.3f}  ->  {beta_keep[1]:+.3f}")
print(f"  sigma               {sig_b:.3f}  ->  {sig_keep:.3f}    ({sig_b / sig_keep:.1f}x smaller)")
print(f"  1.96 sigma, the spread of one station's reading: "
      f"{1.96 * sig_b:.2f}  ->  {1.96 * sig_keep:.2f} log10 units")

The robust fit, given **all** the data including the anomalies, lands close to the
least-squares fit computed **after deleting them**. Nothing was deleted to achieve that; the loss
function was changed.

This is the practical form of A.3's lesson. Choosing squared error is choosing to believe that
residuals are Gaussian, and a Gaussian assigns a deviation of this size a probability of order
$10^{-20}$ — so when one occurs, least squares does not conclude that the point is unusual. It
concludes that the *curve* must be wrong, and moves it. A loss with a linear tail encodes a
different belief about the noise, and the estimate stops depending on whether an analyst
remembered to look for outliers.

The deletion is still the right call **here**, because section 7 established a specific reason to
distrust those two records. Robustness is not a substitute for that. It is what protects the answer
when nobody has checked.

### B.5 · The magnitude, and how well we know it

Now the question the section opened with. Applying the definition of B.2 at each station gives a
**station magnitude** — one estimate of the event's size from that station alone — and the event
magnitude is their average.

The distance correction used is Hutton & Boore's, published for southern California:

$$-\log_{10}A_0(R) = 1.110\,\log_{10}(R/100) + 0.00189\,(R - 100) + 3.0 .$$

**This is a borrowed curve, and borrowing it is a decision with a cost.** Section 7 declined to
compute a magnitude for exactly this reason: the constants of a published attenuation relation are
fitted to one network, one region and one set of instruments. We use it anyway — because B.3 showed
this event cannot determine its own — and then measure what the borrowing costs in B.7 rather than
leaving it unstated.

Then A.5, which is the entire answer to *how well do we know it*: the spread of the station
magnitudes gives both intervals, and they answer different questions.

In [ ]:
def minus_log_A0(R):
    """Hutton & Boore (1987) distance correction, southern California."""
    return 1.110 * np.log10(R / 100.0) + 0.00189 * (R - 100.0) + 3.0

station_mag = np.log10(wa.amp_mean_mm.values) + minus_log_A0(wa.hyp_km.values)
keep_m = station_mag[mask]
n_m = len(keep_m)
M_event, s_station = keep_m.mean(), keep_m.std(ddof=1)

print(f"{n_m} station magnitudes, {keep_m.min():.2f} to {keep_m.max():.2f}")
print(f"\n  event magnitude (their mean)          M = {M_event:.3f}")
print(f"  scatter of ONE station reading        sd = {s_station:.3f}")
print(f"  95% confidence interval on the mean      +/- {1.96 * s_station / np.sqrt(n_m):.3f}")
print(f"  95% prediction interval, one station     +/- {1.96 * s_station:.3f}")
print(f"  ratio {np.sqrt(n_m):.1f}x  -- which is sqrt(n) = sqrt({n_m}), exactly as A.5 derived")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(8.6, 3.6),
                             gridspec_kw={"width_ratios": [1.6, 1]})
a1.errorbar(wa.hyp_km[mask], keep_m, fmt="o", ms=5, mfc="0.75", mec="0.3", mew=0.5)
# The excluded station sits three decades low. Plotted in range it squashes the panel and
# hides the very scatter the figure is about, so it is marked at the axis edge instead.
lo_, hi_ = keep_m.min() - 0.25, max(keep_m.max(), CATALOGUE_MAG) + 0.12
a1.set_ylim(lo_, hi_)
for d_, m_x in zip(wa.hyp_km[~mask], station_mag[~mask]):
    a1.annotate(f"{m_x:.1f}", (d_, lo_), xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=6, color="C3",
                arrowprops=dict(arrowstyle="-|>", color="C3", lw=0.8, shrinkA=0, shrinkB=0))
a1.plot([], [], "v", color="C3", label="excluded (B.4), off scale")
a1.axhline(M_event, color="C0", lw=1.4, label=f"mean, M = {M_event:.2f}")
a1.axhspan(M_event - 1.96 * s_station, M_event + 1.96 * s_station, color="C1", alpha=0.18,
           label="95% prediction interval")
a1.axhspan(M_event - 1.96 * s_station / np.sqrt(n_m), M_event + 1.96 * s_station / np.sqrt(n_m),
           color="C0", alpha=0.45, label="95% confidence interval")
a1.axhline(CATALOGUE_MAG, color="k", ls="--", lw=1.0, label=f"catalogue {CATALOGUE_MAG}")
a1.set_xscale("log"); a1.set_xlabel("hypocentral distance (km)")
a1.xaxis.set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
a1.xaxis.set_minor_formatter(plt.matplotlib.ticker.NullFormatter())
a1.set_xticks([3, 5, 10, 20, 30])
a1.set_ylabel("station magnitude"); a1.legend(fontsize=6.5, loc="lower right")
a1.set_title("one estimate per station")

a2.hist(keep_m, bins=12, color="0.7", edgecolor="0.3")
a2.axvline(M_event, color="C0", lw=1.4)
a2.axvline(CATALOGUE_MAG, color="k", ls="--", lw=1.0)
a2.set_xlabel("station magnitude"); a2.set_ylabel("stations")
a2.set_title("their distribution")
fig.tight_layout(); plt.show()

**Exercise 4.** Most earthquakes in a catalogue are recorded by far fewer than 33 stations
— the median event in this catalogue has 6 contributing to its magnitude.

Resample the station magnitudes with replacement, at every size from 3 up to all of them, 300
resamples each. For each size report the confidence-interval half-width, the prediction-interval half-width,
and the actual standard deviation of the subset means. Two of those three should agree, and one
should stay flat. Say which before running it.

In [ ]:
# your code here


### B.6 · The parameter this dataset cannot estimate

A.7 fitted a site term for each station: a fixed offset saying that this station reads high and that
one low. Those terms carried more than half the scatter in the Part A design, and estimating them
was what the ridge prior was for.

Try it here.

In [ ]:
S = pd.get_dummies(wa.station).values.astype(float)
Xsite = np.column_stack([X, S])
print(f"design with one site term per station: {Xsite.shape[0]} readings, "
      f"{Xsite.shape[1]} columns, rank {np.linalg.matrix_rank(Xsite)}")
print(f"readings per station: max {int(wa.groupby('station').size().max())}")
print()
print("Every station appears exactly once, so there are as many site parameters as readings.")
print("The site terms can reproduce the data perfectly and leave the distance terms with nothing")
print("to explain: the two are not separately identifiable, and no amount of care in the fitting")
print("recovers them. A.7 step 6 is the same statement in linear algebra -- X'X is singular.")

This is the honest boundary of a single-event dataset, and it is worth being precise about
what it does and does not prevent.

It does **not** mean the site effects are absent. The scatter measured in B.5 is a mixture of
radiation pattern, path, and site, and Part A's decomposition of an equivalent scatter put about
half of it in the site term. It means only that *this* dataset cannot say which part is which,
because a site term and a one-off measurement error are the same number when a station is seen once.

Separating them requires seeing each station **many times, at many distances, in many events** — at
which point the site term is what a station shows repeatedly and the noise is what it does not. That
is the only thing the research-scale dataset buys, and it is worth stating plainly: the extra data
does not make the estimator better, it makes a parameter *exist*.

A.7 already showed what to do once it does: penalise the site terms with
$\lambda = \sigma^2/s^2$, the ratio of the within-station scatter to the between-station spread.

### B.7 · What the number is worth

The magnitude determined in B.5 disagrees with the catalogue. That disagreement is not noise, and
three separate decisions contribute to it. Two can be measured here; the third cannot.

In [ ]:
# Each decision applied in turn, starting from the crudest version of the measurement.
def event_mag(amp, dist, drop_outliers=True):
    m = np.log10(amp) + minus_log_A0(dist)
    if drop_outliers:
        Xd = np.column_stack([np.ones(len(dist)), np.log10(dist), dist])
        rd = np.log10(amp) - Xd @ np.linalg.lstsq(Xd, np.log10(amp), rcond=None)[0]
        m = m[rd >= -1.5]
    return m.mean(), len(m)

steps = [("vertical component, outliers kept",
          *event_mag(vert.amp_mm.values, vert.hyp_km.values, drop_outliers=False)),
         ("vertical component, outliers dropped",
          *event_mag(vert.amp_mm.values, vert.hyp_km.values)),
         ("horizontal components, outliers dropped",
          *event_mag(wa.amp_mean_mm.values, wa.hyp_km.values))]

prev = None
print(f"{'':46s} {'M':>7s} {'n':>4s} {'change':>8s}")
for lab, m_, n_ in steps:
    ch = "" if prev is None else f"{m_ - prev:+.3f}"
    print(f"  {lab:44s} {m_:7.3f} {n_:4d} {ch:>8s}")
    prev = m_
print(f"\n  {'catalogue, a coda-duration magnitude':44s} {CATALOGUE_MAG:7.3f}")
print(f"  {'still unaccounted for':44s} {CATALOGUE_MAG - prev:+7.3f}")

The remainder is not a residual to be minimised away. Two named causes sit inside it, and
neither can be measured from one event.

**The distance correction is southern California's.** Hutton & Boore fitted it to a different crust,
a different network and a different instrument population. The Geysers is a geothermal field with
strong attenuation; there is no reason its $-\log_{10}A_0$ should match, and B.3 showed this event
cannot determine its own.

**The catalogue number is a different scale.** It is a coda-duration magnitude, calibrated against
how long the signal stays above the noise — not measured from an amplitude at all. Comparing it with
a local magnitude compares two conventions that were separately tuned to agree *on average, over
some population of earthquakes*, which is not a claim about any single one.

So the notebook stops here, and says so. The comparison is a documented disagreement with two
identified causes and no way to separate them at this scale — which is a more useful thing for a
student to carry than a number that happened to match.

## Part C · The b-value

Part B determined one magnitude. A catalogue holds four hundred thousand of them, and the single
most-quoted number derived from a catalogue is the **b-value**: the slope of the
frequency–magnitude distribution, which says how many small earthquakes there are for each large
one. It enters hazard estimates, aftershock forecasts, and every claim that seismicity has changed.

This part is A.8 applied, and nothing else is new:

| from Part A | used in |
|---|---|
| A.3 the recipe — write the likelihood, maximise, invert the curvature | C.2 |
| A.8 when the noise is not Gaussian | C.3 — least squares on a histogram, on real data |
| A.4 the error bar is the inverse curvature | C.4 — which gives $\sigma_b = b/\sqrt{n}$ exactly |
| A.5 what an interval covers | C.4 — checked against a bootstrap |
| A.4's closing note: at large $n$, $\sigma$ stops being a guide | C.5 — the point the whole session has been building toward |

**On scale.** Part B was one event at demo scale. This is not: four hundred thousand events is the
full catalogue, and the procedure below is the published procedure, unchanged. There is no larger
version of this analysis to graduate to — which makes what C.5 finds harder to dismiss.

### C.1 · The data, and what it is not

The Geysers catalogue, restricted to the producing field — the same window the project notebook's
frequency–magnitude figure uses, so the two can be compared directly.

Two properties of the magnitude column matter before any fitting, and § 7.6 of the project notebook
established both. The column is a **mixture of scales**: overwhelmingly coda-duration `d`, with a
few hundred local `l` and a couple of hundred moment `w`. And the placeholder rows — magnitude
exactly 0.0 with type `Unk` — are not measurements and are removed.

Magnitudes are also **rounded to 0.1** before anything else, which is the convention the project
figure uses and which C.3 will show is not a cosmetic choice.

In [ ]:
URL = ("https://github.com/AI4EPS/EPS207_Observational_Seismology/releases/download/"
       "data-2026fall/geysers_catalog_1969-2026.csv.gz")

def load(url, tries=4):
    """One network call in the whole notebook. Retry, because eight people share one connection."""
    for k in range(tries):
        try:
            return pd.read_csv(url)
        except Exception as exc:
            if k == tries - 1:
                raise
            print(f"  attempt {k + 1} failed ({type(exc).__name__}), retrying")

cat = load(URL)
cat["year"] = cat.time.astype(str).str[:4].astype(int)

FIELD = [-122.99, -122.65, 38.70, 38.899]          # the producing field
sel = cat[cat.longitude.between(FIELD[0], FIELD[1])
          & cat.latitude.between(FIELD[2], FIELD[3])
          & ~cat.magType.isin(["Unk", "MU"])
          & cat.mag.notna() & cat.mag.ne(0.0)].copy()
sel["m"] = np.round(sel.mag, 1)
DM = 0.1

print(f"catalogue           {len(cat):>8,} events")
print(f"in the field window {len(sel):>8,} events, {sel.year.min()}-{sel.year.max()}")
print(f"removed as placeholders or outside the window: {len(cat) - len(sel):,}")
print()
print(sel.magType.value_counts().head(4).to_string())

### C.2 · Why the likelihood is exponential

The Gutenberg–Richter law states that the number of earthquakes at or above magnitude $M$ is

$$\log_{10} N(\ge M) = a - bM .$$

Turn that into a statement about a single earthquake, which is what a likelihood needs.

1. The fraction of events at or above $M$, among those at or above a completeness threshold $M_c$,
   is $N(\ge M)/N(\ge M_c) = 10^{-b(M - M_c)}$. That is a **survival function**.
2. So $P(M' > M) = e^{-\beta (M - M_c)}$ with $\beta = b\ln 10$ — the survival function of an
   **exponential distribution** with rate $\beta$, shifted to start at $M_c$.
3. Its density is $p(M) = \beta e^{-\beta(M - M_c)}$ for $M \ge M_c$.

Now A.3's recipe, with no Gaussian anywhere:

4. **The likelihood.** $\ell(\beta) = n\log\beta - \beta\sum_i (M_i - M_c)$.
5. **Maximise.** $\partial\ell/\partial\beta = n/\beta - \sum_i(M_i - M_c) = 0$, so
   $\hat\beta = 1/(\bar M - M_c)$ and

   $$\hat b = \frac{\log_{10}e}{\bar M - M_c}.$$

   This is Aki's (1965) estimator. It requires **no binning, no histogram and no regression** — just
   the mean magnitude.
6. **The curvature.** $-\partial^2\ell/\partial\beta^2 = n/\beta^2$, so by A.4 the variance of
   $\hat\beta$ is $\beta^2/n$, and propagating to $b$ gives

   $$\sigma_b = \frac{b}{\sqrt{n}} .$$

One correction is needed in practice. Magnitudes are **reported rounded** to $\Delta M = 0.1$, so an
event recorded as $M_c$ was really anywhere in $[M_c - \Delta M/2,\; M_c + \Delta M/2)$. Utsu's
correction replaces $M_c$ by $M_c - \Delta M/2$ in the denominator, and without it $\hat b$ is
biased high.

In [ ]:
def b_mle(m, mc, dm=DM):
    """Aki (1965) with Utsu's binning correction, and the error bar from the curvature."""
    x = np.asarray(m)
    x = x[x >= mc - 1e-9]
    b = np.log10(np.e) / (x.mean() - (mc - dm / 2))
    return b, b / np.sqrt(len(x)), len(x)

def b_lsq(m, mc, dm=DM):
    """The textbook alternative: a straight line through the cumulative log-count."""
    x = np.asarray(m)
    x = x[x >= mc - 1e-9]
    edges = np.round(np.arange(mc, x.max() + dm, dm), 4)
    n = np.array([(x >= e - 1e-9).sum() for e in edges])
    k = n > 0
    A = np.column_stack([np.ones(k.sum()), edges[k]])
    g, *_ = np.linalg.lstsq(A, np.log10(n[k]), rcond=None)
    r = np.log10(n[k]) - A @ g
    se = np.sqrt(np.diag((r @ r / (k.sum() - 2)) * np.linalg.inv(A.T @ A)))[1]
    return -g[1], se, int(k.sum())

m_all = sel.m.values
b0, s0, n0 = b_mle(m_all, 1.2)
print(f"the whole field, M >= 1.2:  b = {b0:.3f} +/- {s0:.3f} from {n0:,} events")
print(f"  computed from one number: the mean magnitude above the cut, {m_all[m_all >= 1.2].mean():.4f}")

### C.3 · Completeness, and the estimator that assumes a straight line

Small earthquakes are missed. Below some magnitude the catalogue stops being a sample of what
happened and becomes a sample of what the network could detect, so the fit must start at a
**completeness magnitude** $M_c$. The standard first estimate is the peak of the binned magnitude
distribution — below the peak, events are being lost — with a small offset added, since the peak
itself is already affected.

With $M_c$ chosen, the two estimators of C.2 can be compared: Aki's, and the one almost every
introductory treatment shows, which plots $\log_{10}N(\ge M)$ against $M$ and fits a straight line
by least squares.

A.8 says what to expect of the second. The counts are Poisson, so their scatter grows with the
count rather than being constant; and each cumulative count contains every count above it, so the
points are strongly correlated rather than independent. Least squares assumes neither.

In [ ]:
def mc_maxc(m, bump=0.2):
    v, n = np.unique(np.round(m, 1), return_counts=True)
    return round(float(v[n.argmax()] + bump), 1)

periods = {"1972-2025 (all)": m_all, "2012-2025": sel[sel.year >= 2012].m.values}

print(f"{'period':>18s} {'Mc':>5s} {'n':>9s} {'b (MLE)':>18s} {'b (least squares)':>19s}")
rows_c = {}
for lab, mm in periods.items():
    mc = mc_maxc(mm)
    bm, sb, n_ = b_mle(mm, mc)
    bl, sl, nb = b_lsq(mm, mc)
    rows_c[lab] = (mc, bm, sb, n_, bl, sl)
    print(f"{lab:>18s} {mc:5.1f} {n_:9,d} {bm:9.3f} +/-{sb:6.3f} {bl:10.3f} +/-{sl:6.3f}")

print()
for lab, (mc, bm, sb, n_, bl, sl) in rows_c.items():
    print(f"  {lab}: least squares differs from the MLE by {100 * (bl - bm) / bm:+.0f} %, "
          f"which is {abs(bl - bm) / sb:.0f} times the MLE's own standard error")

Notice what the two periods do. Least squares is **17 per cent high** in one and **4 per cent
low** in the other: the error does not have a fixed sign, because it depends on how the sparsely
populated large-magnitude bins happen to fall. An estimator that is wrong by an unpredictable
amount is worse than one that is wrong by a known amount, and neither of these disagreements is
visible in the plot.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(8.8, 3.7), sharey=True)
for ax, (lab, mm) in zip(axs, periods.items()):
    mc, bm, sb, n_, bl, sl = rows_c[lab]
    edges = np.round(np.arange(mm.min(), mm.max() + DM, DM), 4)
    cum = np.array([(mm >= e - 1e-9).sum() for e in edges])
    inc, _ = np.histogram(mm, bins=np.append(edges - DM / 2, edges[-1] + DM / 2))
    ax.semilogy(edges[inc > 0], inc[inc > 0], "o", ms=2.4, mfc="none", mec="0.6", mew=0.5,
                label="incremental")
    ax.semilogy(edges[cum > 0], cum[cum > 0], "s", ms=2.6, color="0.25", mew=0, label="cumulative")
    xx = np.arange(mc, mm.max() + DM, DM)
    ax.semilogy(xx, n_ * 10 ** (-bm * (xx - mc)), "-", color="C0", lw=1.4,
                label=f"MLE  b = {bm:.2f}")
    ax.semilogy(xx, n_ * 10 ** (-bl * (xx - mc)), "--", color="C3", lw=1.4,
                label=f"least squares  b = {bl:.2f}")
    ax.axvline(mc, color="0.4", lw=0.6, ls=":")
    ax.set_xlabel("magnitude"); ax.set_title(lab, fontsize=9)
    ax.legend(fontsize=6.5, frameon=False)
axs[0].set_ylabel("number of events")
fig.tight_layout(); plt.show()

In [ ]:
# ── Checkpoint 5 ── run this if you are behind or something broke ──
# Part C state.
FIELD = [-122.99, -122.65, 38.70, 38.899]
DM = 0.1
m_all = sel.m.values
periods = {"1972-2025 (all)": m_all, "2012-2025": sel[sel.year >= 2012].m.values}
rows_c = {lab: (mc_maxc(mm), *b_mle(mm, mc_maxc(mm))[:2], b_mle(mm, mc_maxc(mm))[2],
                *b_lsq(mm, mc_maxc(mm))[:2]) for lab, mm in periods.items()}
print({k: round(v[1], 3) for k, v in rows_c.items()})

### C.4 · Does the error bar mean anything?

$\sigma_b = b/\sqrt{n}$ came out of the curvature of the log-likelihood in C.2, step 6 — the same
route that produced $\sigma^2(X^\top X)^{-1}$ in A.4. It is a claim about repetition, and A.5's
lesson applies: a claim about repetition can be tested by repeating.

There is only one catalogue, so the repetition has to be simulated. Resampling the magnitudes with
replacement and recomputing $\hat b$ each time gives the spread of the estimator directly, with no
formula involved.

**Exercise 5.** Take the 2012–2025 magnitudes above $M_c$. Resample them with replacement 400
times, recompute $\hat b$ for each resample, and compare the spread of those 400 values with
$\sigma_b = b/\sqrt{n}$.

They should agree closely. Note what that does and does not establish: the bootstrap and the formula
both assume the magnitudes are independent draws from one fixed distribution. If that assumption is
wrong, they will agree with each other and both be wrong.

In [ ]:
# your code here


### C.5 · The number the error bar does not cover

A.4 closed with a warning: once $n$ is large, $\sigma$ stops being a guide to whether you have found
anything, because it shrinks as $1/\sqrt n$ while the choices an analyst makes do not shrink at all.

Part C is where that becomes concrete. $\sigma_b$ is three thousandths. Set against it are three
decisions, none of which is wrong, and each of which moves $b$ by more.

In [ ]:
mc_ref = rows_c["1972-2025 (all)"][0]
b_ref, sb_ref = rows_c["1972-2025 (all)"][1], rows_c["1972-2025 (all)"][2]

moves = []
for d in (-0.3, -0.2, 0.2, 0.3, 0.4):
    bb, _, nn = b_mle(m_all, round(mc_ref + d, 1))
    moves.append((f"Mc {mc_ref + d:.1f} instead of {mc_ref:.1f}", bb - b_ref, nn))
moves.append(("least squares instead of the MLE",
              rows_c["1972-2025 (all)"][4] - b_ref, rows_c["1972-2025 (all)"][3]))
moves.append(("no binning correction",
              np.log10(np.e) / (m_all[m_all >= mc_ref].mean() - mc_ref) - b_ref,
              rows_c["1972-2025 (all)"][3]))

print(f"b = {b_ref:.3f},  sigma_b = {sb_ref:.4f}\n")
print(f"{'decision':>42s} {'moves b by':>11s} {'in sigma_b':>11s}")
for lab, dv, nn in moves:
    print(f"{lab:>42s} {dv:+11.3f} {dv / sb_ref:+11.0f}")

In [ ]:
# The same picture as a curve: b against the completeness cut, with the formal interval drawn.
cuts = np.round(np.arange(mc_ref - 0.4, mc_ref + 1.21, 0.1), 1)
bb = np.array([b_mle(m_all, c_)[:2] for c_ in cuts])

fig, ax = plt.subplots(figsize=(6.2, 3.8))
ax.fill_between(cuts, bb[:, 0] - 1.96 * bb[:, 1], bb[:, 0] + 1.96 * bb[:, 1],
                color="C0", alpha=0.35, label=r"95% interval, $\pm 1.96\,\sigma_b$")
ax.plot(cuts, bb[:, 0], "o-", ms=3.5, color="C0", label="b (MLE)")
ax.axvline(mc_ref, color="0.4", lw=0.8, ls=":")
ax.text(mc_ref + 0.02, ax.get_ylim()[1], " Mc from maximum curvature", fontsize=6.5,
        va="top", color="0.4")
ax.set_xlabel("completeness cut used"); ax.set_ylabel("b")
ax.legend(fontsize=7, frameon=False)
ax.set_title("the interval is invisible next to the choice that sets it")
fig.tight_layout(); plt.show()

The band on that figure is the honest 95 per cent interval, computed correctly from the
curvature of the correct likelihood. It is narrower than the line is thick, and the curve it sits on
wanders far outside it as the cut moves across a range of values any analyst might defend.

This is not an argument that the error bar is wrong. It is exactly right about the thing it
describes — the variability of $\hat b$ if these same magnitudes were drawn again from the same
distribution. It is silent about everything else, and everything else is larger.

**Exercise 6.** The project notebook's frequency–magnitude figure reports b = 1.062 for the
whole record and b = 1.108 since 2012, both at a fixed cut of M ≥ 1.2.

Recompute both here at that same cut, with the same estimator and the same binning correction, and
report the difference in units of $\sigma_b$. Before running it, predict whether two careful
analyses of the same catalogue should agree to within the stated uncertainty.

In [ ]:
# your code here


The two analyses differ by ten times the standard error, and the estimator is not the reason
— it is identical in both. The difference is in which events were selected: the window, the
treatment of event types, the handling of the placeholder rows. None of those choices is hidden or
careless, and none of them appears anywhere in $\sigma_b$.

That is the session's last measurement, and it is the reason the first eight sections were about
what an interval means. A number reported as $b = 1.09 \pm 0.003$ is making a precise claim about
one source of variation and no claim at all about the larger ones. Reporting it without saying which
choices were made is how two correct analyses come to disagree in print.

## Takeaways

### Method

- A relation that is linear in its *parameters* is a design matrix, whatever non-linear functions of
  the measurements its columns contain. Choosing those columns is the science; the solve is the same
  every time. Sometimes a coefficient is not fitted at all — Richter **defines** the magnitude
  coefficient to be one.
- Least squares is not a definition. It is maximum likelihood for one noise model, and the object
  worth carrying is the recipe underneath: write the likelihood, maximise it, invert its curvature.
  That recipe produced the normal equations in Part A and Aki's b-value in Part C, which share no
  algebra at all.
- The error bar on any estimate is the inverse curvature of the log-likelihood at its maximum.
- Choosing squared error is choosing to believe the residuals are Gaussian. When a point is far
  outside that belief, least squares moves the curve rather than doubting the point — so one station
  in thirty-three doubled the estimated scatter, and a loss with a linear tail did not notice it.

### Uncertainty

- The interval on *where the curve is* shrinks as more data arrive. The interval on *what the next
  measurement will be* does not. Both were on the same fit, and they differed by a factor of thirty.
- Correlated columns leave the combination determined and the individual coefficients undetermined.
  A published coefficient can disagree with yours by many standard errors while both curves pass
  through the data.
- Ridge is the maximum-a-posteriori estimate under a Gaussian prior, with a penalty that is the
  noise variance over the prior variance. It buys something when data are scarce and nothing when
  they are plentiful — and it is what makes a rank-deficient design solvable at all.
- **A standard error describes one source of variation and is silent about the rest.** As $n$ grows
  it shrinks; the analyst's choices do not. In Part C the choice of completeness cut moved the answer
  by tens of standard errors, and the choice of estimator by more.

### Seismology

- A magnitude is an amplitude plus a distance correction, and the correction carries all the physics.
  Estimating one from a single event does not work: the two distance terms trade against each other,
  and a site term cannot be told from a measurement error when a station is seen once
  (Richter 1935, `10.1785/bssa0250010001`; Hutton & Boore 1987, `10.1785/bssa0770062074`).
- Borrowing a distance correction from another region is a decision with a cost, and the project
  notebook's § 7 declines to make it silently for that reason. Here it was made openly and the cost
  measured: against a coda-duration catalogue magnitude, half a unit remained unexplained after the
  component choice and the outliers were accounted for.
- Above the completeness magnitude, earthquake magnitudes are exponentially distributed, so the
  b-value has a closed-form maximum-likelihood estimator needing only the mean magnitude
  (Aki 1965; the binning correction is Utsu's). Fitting a line to the cumulative log-histogram
  instead is biased in a direction that changes between periods.
- The magnitude column of this catalogue is a mixture of scales, overwhelmingly coda duration, so a
  b-value fitted across its full range is fitted across a change of scale — as § 7.6 of the project
  notebook sets out.